In [2]:
import nltk.corpus
from collections import Counter
from nltk.util import bigrams
from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures

nltk.download('brown')
brown = nltk.corpus.brown
bwords = brown.words()

[nltk_data] Downloading package brown to /Users/myriam/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.


In [6]:
def calculate_pmi(bwords, only_positive=False, freq_threshold=10):
    word_freq = Counter(bwords)
    bigram_freq = Counter(bigrams(bwords))

    # Words below the frequency threshold are too rare to give reliable
    # co-occurrence statistics
    valid_words = {word for word, freq in word_freq.items() if freq >= freq_threshold}
    filtered_bigram_freq = {
        bigram: freq 
        for bigram, freq in bigram_freq.items() 
        if bigram[0] in valid_words and bigram[1] in valid_words
    }

    # PMI = log2(P(w1,w2) / (P(w1)*P(w2)))
    # Measures how much more (or less) often two words co-occur than
    # we'd expect if they were independent
    bigram_finder = BigramCollocationFinder.from_words(bwords)
    bigram_finder.apply_word_filter(lambda w: word_freq[w] < freq_threshold)
    all_pmi_scores = {
        bigram: bigram_finder.score_ngram(BigramAssocMeasures.pmi, bigram[0], bigram[1])
        for bigram in filtered_bigram_freq
        if bigram_finder.score_ngram(BigramAssocMeasures.pmi, bigram[0], bigram[1]) is not None
    }
    
    # PPMI clamps negative values to 0, keeps only positive associations
    if only_positive:
        all_pmi_scores = {bigram: max(pmi, 0) for bigram, pmi in all_pmi_scores.items()}
    sorted_pmi = sorted(all_pmi_scores.items(), key=lambda x: x[1], reverse=True)
        
    return sorted_pmi

In [8]:
#PMI on full Brown corpus
sorted_pmi = calculate_pmi(bwords, only_positive=False)

print("=== Top 20 Bigrams by PMI ===")
for bigram, pmi in sorted_pmi[:20]:
    print(f"  {bigram}: {pmi:.4f}")

print("\n=== Bottom 20 Bigrams by PMI ===")
for bigram, pmi in sorted_pmi[-20:]:
    print(f"  {bigram}: {pmi:.4f}")

=== Top 20 Bigrams by PMI ===
  ('Hong', 'Kong'): 16.6877
  ('Viet', 'Nam'): 16.1472
  ('Pathet', 'Lao'): 16.0597
  ('Simms', 'Purdew'): 16.0597
  ('El', 'Paso'): 15.8252
  ('7th', 'Cavalry'): 15.8252
  ('Herald', 'Tribune'): 15.7947
  ('Lo', 'Shu'): 15.7549
  ('WTV', 'antigen'): 15.6691
  ('Gray', 'Eyes'): 15.6326
  ('Puerto', 'Rico'): 15.5622
  ('Internal', 'Revenue'): 15.5622
  ('decomposition', 'theorem'): 15.3398
  ('Saxon', 'Shore'): 15.3143
  ('anionic', 'binding'): 15.3107
  ('Export-Import', 'Bank'): 15.2892
  ('carbon', 'tetrachloride'): 15.2618
  ('Common', 'Market'): 15.2403
  ('unwed', 'mothers'): 15.2403
  ('Beverly', 'Hills'): 15.2243

=== Bottom 20 Bigrams by PMI ===
  ('in', 'of'): -7.6606
  ('of', 'he'): -7.6725
  ('he', 'of'): -7.6725
  ('to', 'was'): -7.7593
  ('the', 'not'): -7.9001
  ('?', 'the'): -7.9856
  ('of', 'for'): -8.1017
  ('the', 'I'): -8.1227
  (',', ';'): -8.1273
  ('of', 'to'): -8.6430
  ('the', 'and'): -8.9731
  ('the', 'is'): -9.0786
  ('and', 'and'

The implementation of PMI on the Brown corpus has expected results. The highest scoring pair is ('Hong', 'Kong'), this is expected since it is a so called split word. The words have very little meaning on their own, only as a pair is when they make sense. Additionally the words have no other meaning in the English language except for the reference to the country Hong Kong. The high PMI is due to the low usage of the words not as a pair.

The lowest scoring word pairs on PMI consist mainly of stop words that are grammatically incorrect when the order is wrong. The lowest scoring pair is ('.', ','); which is coherent with the English grammar, just like ('the', 'I') and ('the', 'and'). 

The PMI on the corpus proves to be valid however, the results are biased to higher scores for split-words and lower from stop words. Further research should investigate the implementation with preprocessed data that remove these biases. 

In [9]:
#Compare PMI and PPMI on both brown100 and full Brown.
# For brown100, freq_threshold is lowered to 3 because the corpus is small
# (~2500 tokens) and threshold=10 would filter out almost everything.

with open('./brown_100.txt', 'r') as f:
    brown100_text = f.read()
brown100_words = brown100_text.lower().split()

def compare_pmi_ppmi(bwords, label, freq_threshold=10):
    pmi_scores = calculate_pmi(bwords, only_positive=False, freq_threshold=freq_threshold)
    ppmi_scores = calculate_pmi(bwords, only_positive=True, freq_threshold=freq_threshold)
    
    print(f"\n=== {label}: Top 20 by PMI ===")
    for (bigram, pmi) in pmi_scores[:20]:
        ppmi = max(pmi, 0)
        print(f"  {bigram}: PMI={pmi:.4f}, PPMI={ppmi:.4f}")
    
    print(f"\n=== {label}: Bottom 20 by PMI ===")
    for (bigram, pmi) in pmi_scores[-20:]:
        ppmi = max(pmi, 0)
        print(f"  {bigram}: PMI={pmi:.4f}, PPMI={ppmi:.4f}")

# brown100 is small so we lower the threshold
compare_pmi_ppmi(brown100_words, "Brown100", freq_threshold=3)
compare_pmi_ppmi(bwords, "Full Brown", freq_threshold=10)


=== Brown100: Top 20 by PMI ===
  ('million', 'worth'): PMI=8.9472, PPMI=8.9472
  ('school', 'superintendent'): PMI=8.9472, PPMI=8.9472
  ('rural', 'roads'): PMI=8.6842, PPMI=8.6842
  ('bond', 'issue'): PMI=8.3622, PPMI=8.3622
  ('(', '2'): PMI=8.0992, PPMI=8.0992
  ('2', ')'): PMI=8.0992, PPMI=8.0992
  ('elected', 'texas'): PMI=8.0992, PPMI=8.0992
  ('out', 'into'): PMI=8.0992, PPMI=8.0992
  ('more', 'term'): PMI=8.0992, PPMI=8.0992
  ('anonymous', 'calls'): PMI=8.0992, PPMI=8.0992
  ('had', 'been'): PMI=7.8768, PPMI=7.8768
  ('highway', 'bond'): PMI=7.8768, PPMI=7.8768
  ('legislators', 'act'): PMI=7.6842, PPMI=7.6842
  ('also', 'recommended'): PMI=7.6842, PPMI=7.6842
  ('(', '1'): PMI=7.6842, PPMI=7.6842
  ('1', ')'): PMI=7.6842, PPMI=7.6842
  ('petition', 'listed'): PMI=7.6842, PPMI=7.6842
  ('one', 'more'): PMI=7.6842, PPMI=7.6842
  ('or', '2'): PMI=7.6842, PPMI=7.6842
  ('primary', 'under'): PMI=7.6842, PPMI=7.6842

=== Brown100: Bottom 20 by PMI ===
  (',', 'a'): PMI=0.6398, PP

PPMI shows almost no difference in scores on brown100 since nearly all PMI scores are positive (only five pairs are slightly negative). On the full Brown corpus, the difference is clear: the lowest scoring bigrams have strongly negative PMI (e.g. ('.', ',') at -11.28) but PPMI sets them all to 0. The top-scoring bigrams are identical across PMI and PPMI since they're already positive. The downside of PPMI is that it loses the distinction between pairs that weakly and strongly repel each other — ('.', ',') at -11.28 and ('in', 'of') at -7.66 both become 0.